# E31 --- o orçamento que sobrevive a olhar sempre

O capítulo do orçamento declara o vigia com um limiar escolhido **entre seis**, olhando os tombos do
mercado, e imprime a raridade do falso alarme do vencedor. Duas perguntas ficam de pé, e as duas são
sobre o instrumento, não sobre o mundo: aquele orçamento é **do limiar** ou **do procedimento que o
escolheu**? E ele sobrevive a ser consultado **todo dia**? Esta medição responde as duas --- a
primeira medindo a escolha dentro de mundos sorteados, a segunda trocando o limiar por um capital.


In [1]:
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import aposta, dados, graficos, promessa, vigia, volatilidade

RAIZ = Path.cwd()
SERIE = "sp500.csv"
JANELA, CAUDA = 252, 0.05
BLOCO = 60
LIMIARES = (8, 9, 10, 11, 12, 13)
LIMIAR_ESCOLHIDO = 13
MUNDOS_PARADO = 200
MUNDOS_TOMBO = 60
DIAS_TOMBO = 20
QUEDA_DIARIA = 0.02
SEMENTE = 53
ALFA = 0.05

preco = dados.carregar_serie(SERIE)
retornos = volatilidade.retornos_log(preco)
contagem = promessa.conta_em_blocos(promessa.violacoes(retornos, JANELA, CAUDA), BLOCO)
print("frevolab %s | %d dias | %d blocos de %d | limiares %s"
      % (frevolab.VERSAO, retornos.size, contagem.size, BLOCO, LIMIARES))


frevolab 0.1.0 | 6718 dias | 6407 blocos de 60 | limiares (8, 9, 10, 11, 12, 13)


In [2]:
# Painel 1 --- o orcamento e do limiar ou do procedimento que o escolheu?
sorteio = np.random.default_rng(SEMENTE)
blocos_nulos = np.empty((MUNDOS_PARADO, contagem.size))
precos_nulos = np.empty((MUNDOS_PARADO, retornos.size))
for i in range(MUNDOS_PARADO):
    mundo = pd.Series(sorteio.normal(0.0, 0.01, retornos.size), index=retornos.index)
    blocos_nulos[i] = promessa.conta_em_blocos(promessa.violacoes(mundo, JANELA, CAUDA), BLOCO)
    precos_nulos[i] = 100.0 * np.exp(mundo.cumsum())
orcamentos = {t: vigia.orcamento(blocos_nulos, t) for t in LIMIARES}
print("%6s %18s %16s" % ("limiar", "anos por alarme", "mundos que soam"))
for t in LIMIARES:
    o = orcamentos[t]
    print("%6d %18.1f %15.1f%%" % (t, o["anos_por_alarme"], 100 * o["fracao_com_alarme"]))

# A escolha, dentro de mundos sorteados com um tombo injetado: a regra do capitulo e "o limiar que
# chega mais cedo ao maior tombo", isto e, o de menor prejuizo ja pago no primeiro alarme.
escolhidos, pagos = [], []
for i in range(MUNDOS_TOMBO):
    r = sorteio.normal(0.0, 0.01, retornos.size)
    comeco = int(sorteio.integers(400, retornos.size - 400))
    r[comeco:comeco + DIAS_TOMBO] -= QUEDA_DIARIA
    mundo = pd.Series(r, index=retornos.index)
    preco_mundo = pd.Series(100.0 * np.exp(mundo.cumsum()), index=retornos.index)
    conta = promessa.conta_em_blocos(promessa.violacoes(mundo, JANELA, CAUDA), BLOCO)
    melhor, pior = None, None
    for t in LIMIARES:
        rotulos = [e["inicio"] for e in vigia.alarmes(conta, t)]
        posicoes = [conta.index.get_loc(r) for r in rotulos]
        depois = [r for p_, r in zip(posicoes, rotulos) if p_ >= comeco - 5]
        if not depois:
            continue
        pago = vigia.prejuizo_pago(preco_mundo, depois[0])["fracao_paga"]
        if melhor is None or pago < melhor:
            melhor, pior = pago, t
    if pior is not None:
        escolhidos.append(pior); pagos.append(melhor)
escolhidos = np.array(escolhidos)
print()
print("em %d mundos com tombo, o limiar vencedor se distribui assim:" % escolhidos.size)
for t in LIMIARES:
    quantos = int((escolhidos == t).sum())
    if quantos:
        print("   limiar %2d: %3d mundos (%.0f%%) --- orcamento de %.1f anos por alarme"
              % (t, quantos, 100 * quantos / escolhidos.size, orcamentos[t]["anos_por_alarme"]))
vencedor = float(np.mean([orcamentos[t]["anos_por_alarme"] for t in escolhidos]))
print("orcamento medio do limiar VENCEDOR: %.1f anos por alarme" % vencedor)
print("orcamento do limiar 13, o que o livro imprime: %.1f anos por alarme"
      % orcamentos[LIMIAR_ESCOLHIDO]["anos_por_alarme"])


limiar    anos por alarme  mundos que soam
     8                4.0            99.0%
     9               11.6            82.0%
    10               37.7            41.0%
    11              158.9            12.5%
    12              635.6             3.0%
    13             5084.9             0.5%



em 52 mundos com tombo, o limiar vencedor se distribui assim:
   limiar  8:  45 mundos (87%) --- orcamento de 4.0 anos por alarme
   limiar  9:   5 mundos (10%) --- orcamento de 11.6 anos por alarme
   limiar 10:   1 mundos (2%) --- orcamento de 37.7 anos por alarme
   limiar 11:   1 mundos (2%) --- orcamento de 158.9 anos por alarme
orcamento medio do limiar VENCEDOR: 8.4 anos por alarme
orcamento do limiar 13, o que o livro imprime: 5084.9 anos por alarme


In [3]:
# Painel 2 --- o capital: o orcamento que vale para todos os instantes.
sorteio = np.random.default_rng(SEMENTE + 1)
blocos_parados = np.array([aposta.contagens_por_bloco(
    promessa.violacoes(pd.Series(sorteio.normal(0.0, 0.01, retornos.size), index=retornos.index),
                       JANELA, CAUDA), BLOCO) for _ in range(MUNDOS_PARADO)])
limiar_capital = aposta.orcamento_de_ville(ALFA)
caminhos = aposta.capital(blocos_parados, BLOCO, eixo=1)
soam = float((caminhos.max(axis=1) >= limiar_capital).mean())
print("blocos que nao se sobrepoem: %d por mundo de %.1f anos" % (blocos_parados.shape[1], retornos.size / 252.0))
print("capital que assina %.0f%% em qualquer instante: %.0f" % (100 * ALFA, limiar_capital))
print("mundos parados que cruzam em algum instante: %.2f%% (Ville promete no maximo %.0f%%)"
      % (100 * soam, 100 * ALFA))
print("capital tipico no fim: mediana %.4f | media %.4f --- o tipico cai, e a cauda paga"
      % (float(np.median(caminhos[:, -1])), float(caminhos[:, -1].mean())))

# A latencia no mundo que MUDA: a taxa passa de 5% para 10% no meio da serie.
metade = blocos_parados.shape[1] // 2
mudados = np.concatenate([sorteio.binomial(BLOCO, CAUDA, size=(MUNDOS_PARADO, metade)),
                          sorteio.binomial(BLOCO, 0.10, size=(MUNDOS_PARADO, metade))], axis=1)
caminhos_mudados = aposta.capital(mudados, BLOCO, eixo=1)
cruzamentos = [aposta.primeiro_cruzamento(c, limiar_capital) for c in caminhos_mudados]
atrasos = [c["indice"] - metade for c in cruzamentos if c["cruzou"] and c["indice"] >= metade]
print("dos %d mundos que mudaram, %d cruzam depois da mudanca; atraso mediano %.0f blocos (%d dias)"
      % (MUNDOS_PARADO, len(atrasos), float(np.median(atrasos)), int(np.median(atrasos)) * BLOCO))


blocos que nao se sobrepoem: 107 por mundo de 26.7 anos
capital que assina 5% em qualquer instante: 20
mundos parados que cruzam em algum instante: 1.00% (Ville promete no maximo 5%)
capital tipico no fim: mediana 0.0000 | media 0.0000 --- o tipico cai, e a cauda paga
dos 200 mundos que mudaram, 137 cruzam depois da mudanca; atraso mediano 39 blocos (2340 dias)


In [4]:
# Figura 1: o capital tipico contra o limiar, num mundo parado e num que muda.
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(9.6, 3.8))
for c in caminhos[:60]:
    esq.plot(c, lw=0.7, color="#1f4e79", alpha=0.35)
esq.axhline(limiar_capital, color="#b03a2e", lw=1.4, ls="--", label="o limiar (1/alfa)")
esq.set_yscale("log")
esq.set_title("mundo que nunca muda", fontsize=10)
esq.set_xlabel("blocos"); esq.set_ylabel("capital")
esq.legend(frameon=False, fontsize=8); esq.grid(alpha=0.25, ls=":", which="both")
for c in caminhos_mudados[:60]:
    dir_.plot(c, lw=0.7, color="#2e7d32", alpha=0.35)
dir_.axvline(metade, color="#555555", lw=1.0, ls=":")
dir_.axhline(limiar_capital, color="#b03a2e", lw=1.4, ls="--")
dir_.set_yscale("log")
dir_.set_title("mundo que muda no meio", fontsize=10)
dir_.set_xlabel("blocos"); dir_.grid(alpha=0.25, ls=":", which="both")
fig.tight_layout()
graficos.salvar(fig, "E31_aposta", 1)
plt.close(fig)
print("figura gravada")


figura gravada


## Leitura visual das figuras

Feita nesta sessão abrindo o `E31_aposta_1.png` com a ponte de visão (AGENTS.md §9), depois de o
caderno rodar, e conferida contra o `E31_aposta.json`.

**O que o desenho mostra.** Dois painéis, ambos com o capital no eixo vertical em escala logarítmica
e os blocos no horizontal, e em ambos a mesma linha tracejada alta, que é o limiar de um sobre alfa.

**À esquerda**, o mundo que nunca muda: as trajetórias **caem**, e caem muito --- o eixo desce até
10⁻⁴⁸ ---, e só algumas cruzam a linha. É o que uma martingala é: o capital típico se perde, e é a
cauda que paga.

**À direita**, o mundo que muda no meio: a linha pontilhada vertical marca a mudança, e as
trajetórias passam a **subir** depois dela, com a cauda chegando a 10³².

**E os números concordam.** 1,00% dos mundos parados cruzam o limiar (Ville promete no máximo 5%), a
mediana do capital no fim é praticamente zero, e o atraso mediano no mundo que muda é de 39 blocos.


In [5]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {8: "oito", 9: "nove", 10: "dez", 11: "onze", 12: "doze", 13: "treze"}
resultado = {
    "aposta_mundos_parado": int(MUNDOS_PARADO),
    "aposta_mundos_tombo": int(MUNDOS_TOMBO),
    "aposta_blocos": int(blocos_parados.shape[1]),
    "aposta_semente": int(SEMENTE),
    "aposta_alfa": 100 * float(ALFA),
    "aposta_capital_limiar": int(limiar_capital),
    "aposta_soam_pct": round(100 * soam, 2),
    "aposta_capital_mediano_pct": round(100 * float(np.median(caminhos[:, -1])), 2),
    "aposta_latencia_blocos": int(np.median(atrasos)),
    "aposta_latencia_dias": int(np.median(atrasos)) * BLOCO,
    "aposta_orcamento_vencedor": round(vencedor, 1),
    "aposta_orcamento_escolhido": round(float(orcamentos[LIMIAR_ESCOLHIDO]["anos_por_alarme"]), 1),
    "aposta_limiares": int(len(LIMIARES)),
    "aposta_tombo_dias": int(DIAS_TOMBO),
}
for t in LIMIARES:
    resultado["aposta_anos_%s" % NOMES[t]] = round(float(orcamentos[t]["anos_por_alarme"]), 1)
    resultado["aposta_vencedor_%s" % NOMES[t]] = int((escolhidos == t).sum())
caminho = Path("lab/resultados/E31_aposta.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E31_aposta.json gravado | 26 grandezas
